In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import time
import os

load_dotenv()

In [ ]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
GROQ_MODEL=os.getenv("GROQ_MODEL")

llm = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY)

In [ ]:
# defien state

class CrashState(TypedDict):
    input: str
    state_1: str
    state_2: str
    state_3: str

In [ ]:
def state_1(state: CrashState):
    
    input = state["input"]
    
    print("Step 1 is executed")
    
    return {"state_1": "done", "input": input}

In [ ]:
def state_2(state: CrashState):
    
    print("Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    
    time.sleep(1000)
    
    return {"state_2": "done"}

In [ ]:
def state_3(state: CrashState):
    
    print("Step 3 executed")
    
    return {"state_3": "done"}

In [ ]:
graph = StateGraph(CrashState)

graph.add_node("state_1", state_1)
graph.add_node("state_2", state_2)
graph.add_node("state_3", state_3)

graph.set_entry_point("state_1")
graph.add_edge("state_1", "state_2")
graph.add_edge("state_2", "state_3")
graph.add_edge("state_3", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

workflow

In [ ]:
try:
    print("Running graph: Please manually interrupt during Step 2...")
    workflow.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("Kernel manually interrupted (crash simulated).")

In [ ]:
print("\nRe-running the graph to demonstrate fault tolerance...")
final_state = workflow.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\nFinal State:", final_state)